# Comparing Synthetic vs Real

1. Marginal distributions (KS & $\chi^2$ Tests)
    Compare output distributions of peaks ATACseq peaks/RNAseq peaks
2. Pairwise correlation
    Compare ATACseq and RNA for real, compare with synthetic data
3. Dimensionality Reduction
    Plot PCA/UMAP for generated and real data
4. Classification
    Train classifier on synthetic data and Real data. Determine if it can correctly distinguish between the two.

I have also added a env.yaml file for creating the conda environment for the workspace. To create it, run the following.
```
conda create -n analysis_env -f path/to/env.yaml
```

Questions?
Is the data trained on normalized counts already or will normalization need to be applied after training to compare the two?

## Import Libraries and data

In [ ]:
import muon as mu
from muon import atac as ac
import scanpy as sc
import anndata as ad
from scipy.stats import ks_2samp, chi2_contingency

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

real_path = ""
#gen_path="Add when Finished"

mdata_real = mu.read_h5ad(real_path, mod=None)
#mdata_gen = mu.read_h5ad(gen_path)

#Remove all this once we have actual files to use. This is just processing to ensure functions are working correctly. 
adata_real = mu.read_h5ad(real_path, None)
rna = adata_real[:, adata_real.var["feature_types"] == "GEX"].copy()
atac = adata_real[:, adata_real.var["feature_types"] == "ATAC"].copy()

mdata_real = mu.MuData({"rna": rna, "atac":atac})

In [ ]:
#Temporary cell to generate synthetic data. 
import numpy as np
np.random.seed = 67
n_cells = mdata_real.n_obs
indices = np.random.choice(mdata_real.obs_names, size=int(n_cells*0.2), replace=False)
mdata_gen = mdata_real[indices, :].copy()


# 1 Marginal Distribution 
(Mmmmmm, margarine)

In [ ]:
def ks_divergence(real_data, gen_data, method, obs_key):
    """
    Method= ['atac' or 'rna']
    """ 
    vec_real=real_data[method].obs[obs_key].values   
    vec_gen = gen_data[method].obs[obs_key].values
    ks_stat, p_val = ks_2samp(vec_real, vec_gen)
    return ks_stat, p_val

# Now call the function
print(ks_divergence(mdata_real, mdata_gen, "atac", "GEX_n_counts"), 
      ks_divergence(mdata_real, mdata_gen, "rna", "ATAC_nCount_peaks"))

This can be rewritten as well. In the example dataset Aidana sent me this was the best way to access it, but in the examples they use a "rna:n_counts" or "atac:atac_nCount_peaks" within the obs field ```(vec = data.obs["rna:obs_key"].values)```. It doesn't work with this testing dataset, but maybe the other data is formatted correctly. 

# 2 Pairwise Correlation
This one has been running into the most issues in implementing. I opted to use PCA to find the features to plot, but its also possible to use highly variable genes. The issue is that I could not get the function calling the highly variable genes to function properly. 

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

combined_rna = mdata_real["rna"].concatenate(mdata_gen["rna"], batch_categories=["Real", "Synthetic"])

# 1. Ensure PCA is run
sc.tl.pca(combined_rna, n_comps=50)

# 2. Extract the top 20 genes contributing to PC1 and PC2
pc_loadings = pd.DataFrame(combined_rna.varm['PCs'][:, :2], index=combined_rna.var_names, columns=['PC1', 'PC2'])
top_genes = pc_loadings.abs().sort_values('PC1', ascending=False).index[:30]

In [ ]:
def plot_corr_heatmap(adata, gene_list, title="Correlation Matrix"):
    # Extract data for these genes
    data = adata[:, gene_list].X.toarray()
    df = pd.DataFrame(data, columns=gene_list)
    
    # Compute Pearson correlation
    corr_matrix = df.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False, vmin=-1, vmax=1)
    plt.title(title)
    plt.show()

# Run for both to compare "patterns"
plot_corr_heatmap(mdata_real['rna'], top_genes, "Real Data Correlations")
plot_corr_heatmap(mdata_gen['rna'], top_genes, "Synthetic Data Correlations")

In [ ]:
# Flatten the upper triangle of the correlation matrices (to avoid double counting)
real_corr = pd.DataFrame(mdata_real['rna'][:, top_genes].X.toarray()).corr().values
synth_corr = pd.DataFrame(mdata_gen['rna'][:, top_genes].X.toarray()).corr().values

iu = np.triu_indices(len(top_genes), k=1)
correlation_of_corrs = np.corrcoef(real_corr[iu], synth_corr[iu])[0, 1]

print(f"Structural Correlation Score: {correlation_of_corrs:.4f}")

## Adjusted (Ayn) Rand Score

In [ ]:
from sklearn.metrics import adjusted_rand_score as ari

ari(mdata_gen["rna"].obs["cell_type"], mdata_gen["atac"].obs["cell_type"]) #only subset of data, makes sense that it doesn't show anything now. But may be useful for later analysis. 

# Dimensionality Reduction
Set up for PCA and UMAP visualizations. 

In [ ]:
combined_rna = mdata_real["rna"].concatenate(mdata_gen["rna"], batch_categories=["Real", "Synthetic"])
combined_atac= mdata_real["atac"].concatenate(mdata_gen["atac"], batch_categories=["Real", "Synthetic"])

sc.pp.pca(combined_rna)
sc.pp.neighbors(combined_rna)
sc.tl.umap(combined_rna)
sc.pl.umap(combined_rna, color='batch', title="Manifold Overlap RNA")

sc.pp.pca(combined_atac)
sc.pp.neighbors(combined_atac)
sc.tl.umap(combined_atac)
sc.pl.umap(combined_atac, color='batch', title="Manifold Overlap ATAC")

In [ ]:
sc.pp.pca(combined_rna)
sc.pp.neighbors(combined_rna)
sc.tl.pca(combined_rna)
sc.pl.pca(combined_rna, color='batch', title="PCA Overlap RNA")

sc.pp.pca(combined_atac)
sc.pp.neighbors(combined_atac)
sc.tl.pca(combined_atac)
sc.pl.pca(combined_atac, color='batch', title="PCA Overlap ATAC")

The PCAs look quite beautiful. 
# Classifier Training

Train classifier on real and synthetic data. Test its recall

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, classification_report

combined_rna.obs['is_synthetic_label'] = combined_rna.obs['batch'].map({
    "Real": 1, 
    "Synthetic": 0
}).astype(int)
adata_combined = combined_rna.copy()

# Features X and Labels y
X = adata_combined.X.toarray() if hasattr(adata_combined.X, "toarray") else adata_combined.X
y = adata_combined.obs["is_synthetic_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=67, stratify=y
)

In [ ]:
clf=RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=67)
clf.fit(X_train, y_train)

probs =clf.predict_proba(X_test)[:,1]
preds =clf.predict(X_test)

In [ ]:
# 1. Calculate ROC-AUC
roc_score = roc_auc_score(y_test, probs)

# 2. Calculate Precision-Recall AUC
precision, recall, _ = precision_recall_curve(y_test, probs)
pr_auc = auc(recall, precision)

print(f"Discriminator ROC-AUC: {roc_score:.4f}")
print(f"Discriminator PR-AUC: {pr_auc:.4f}")
print("\nDetailed Report:\n", classification_report(y_test, preds))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Plot ROC Curve
RocCurveDisplay.from_estimator(clf, X_test, y_test, ax=ax[0], color='darkorange')
ax[0].plot([0, 1], [0, 1], "k--", label="Random Chance (Perfect GNN)")
ax[0].set_title("Discriminator ROC Curve")
ax[0].legend()

# Plot Precision-Recall Curve
PrecisionRecallDisplay.from_estimator(clf, X_test, y_test, ax=ax[1], color='blue')
ax[1].set_title("Discriminator PR Curve")

plt.tight_layout()
plt.show()